# Module 4 - Topic 2: Types of Machine Learning
## Live Demo Notebook

**Scenario:** A Nigerian telecoms company wants to understand its customer base.
We have two datasets:
1. **Labelled data** - 50 customers with a `churned` column → **Supervised Learning**
2. **Unlabelled data** - 50 customers with only spending behaviour → **Unsupervised Learning**

We'll run both approaches and see the difference in how they work and what they produce.

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded.')

---
## Part 1 - Supervised Learning: Predicting Customer Churn

We have **labelled data** - each customer record includes whether they churned (left the network).
Goal: train a model to predict which customers will churn next.

In [ ]:
# Load labelled data
churn_df = pd.read_csv('customer_churn.csv')
print('Labelled dataset shape:', churn_df.shape)
print()
print('Notice the last column: churned = 1 (left) or 0 (stayed)')
print('This is the LABEL - the correct answer recorded in history.')
print()
churn_df.head(8)

In [ ]:
# Separate features from label
X = churn_df[['age', 'monthly_spend', 'num_complaints']]
y = churn_df['churned']

print(f'Features (X): {X.shape[1]} columns - what we know about the customer')
print(f'Label    (y): 1 column - did they churn?')
print(f'Churn rate:   {y.mean()*100:.0f}% of customers churned')

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'\nTraining: {len(X_train)} customers | Test: {len(X_test)} customers')

In [ ]:
# Train a supervised classifier
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)   # <-- learning from labelled data

# Predict on unseen test customers
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print('Supervised Learning - Churn Prediction Results:')
print(f'Test accuracy: {accuracy*100:.0f}%')
print()
print('Predictions vs Actual:')
for actual, predicted in zip(y_test, y_pred):
    a = 'CHURNED' if actual    else 'STAYED '
    p = 'CHURNED' if predicted else 'STAYED '
    match = '✓' if actual == predicted else '✗'
    print(f'  Actual: {a} | Predicted: {p} | {match}')

In [ ]:
# Predict on a new customer the model has never seen
new_customers = pd.DataFrame({
    'age':            [28,    45,    33   ],
    'monthly_spend':  [1200,  12000, 5500 ],
    'num_complaints': [5,     0,     2    ],
})

predictions = clf.predict(new_customers)
print('Predictions for 3 new customers:')
print()
for i, (_, row) in enumerate(new_customers.iterrows()):
    result = 'WILL CHURN' if predictions[i] else 'WILL STAY'
    print(f'  Customer {i+1}: age={row.age}, spend=₦{row.monthly_spend:,}, complaints={row.num_complaints}')
    print(f'  Prediction: {result}')
    print()

---
## Part 2 - Unsupervised Learning: Discovering Customer Segments

Now we have **unlabelled data** - only spending behaviour, no `churned` column.
Goal: let the algorithm discover natural customer groups by itself.

In [ ]:
# Load unlabelled data
spend_df = pd.read_csv('customer_spending.csv')
print('Unlabelled dataset shape:', spend_df.shape)
print()
print('Notice: NO label column. We only have measurements.')
print('There is no "correct answer" to predict - we want to discover groups.')
print()
spend_df.head(8)

In [ ]:
# Plot the raw data - do you see any natural groups?
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(spend_df['monthly_spend'], spend_df['num_transactions'],
           color='grey', alpha=0.7, s=80, edgecolors='white')
ax.set_title('Customer Spending Data - Can You See Natural Groups?',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Monthly Spend (₦)')
ax.set_ylabel('Number of Transactions')
plt.tight_layout()
plt.show()
print('Before running any algorithm - do you see clusters in this chart?')

In [ ]:
# Run K-Means clustering - tell it to find 3 groups
X_spend = spend_df[['monthly_spend', 'num_transactions']]

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_spend)   # <-- no labels passed! Only X, not y.

# Add cluster assignments back to the DataFrame
spend_df['cluster'] = kmeans.labels_

print('K-Means found 3 clusters:')
print(spend_df['cluster'].value_counts().sort_index())
print()
print('Cluster summary (mean values per group):')
print(spend_df.groupby('cluster')[['monthly_spend', 'num_transactions']].mean().round(0))

In [ ]:
# Plot the same data - now coloured by discovered cluster
cluster_names = {0: 'Segment A', 1: 'Segment B', 2: 'Segment C'}
colours = ['#E63946', '#2A9D8F', '#E9C46A']

fig, ax = plt.subplots(figsize=(9, 5))
for cluster_id in sorted(spend_df['cluster'].unique()):
    subset = spend_df[spend_df['cluster'] == cluster_id]
    ax.scatter(subset['monthly_spend'], subset['num_transactions'],
               color=colours[cluster_id], s=90, edgecolors='white',
               label=f'Cluster {cluster_id}')

# Plot centroids
centers = kmeans.cluster_centers_
ax.scatter(centers[:, 0], centers[:, 1], marker='X', s=200, color='black',
           zorder=5, label='Centroids')

ax.set_title('K-Means Discovered 3 Natural Customer Segments',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Monthly Spend (₦)')
ax.set_ylabel('Number of Transactions')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Interpret the clusters - what do they mean for the business?
summary = spend_df.groupby('cluster')[['monthly_spend', 'num_transactions']].mean().round(0)
summary = summary.sort_values('monthly_spend', ascending=False)

print('=== Cluster Interpretation ===')
for cluster_id, row in summary.iterrows():
    spend = row['monthly_spend']
    txns  = row['num_transactions']
    if spend > 8000:
        label = 'VIP Customers - high spend, frequent transactions'
        action = 'Offer premium plans, loyalty rewards'
    elif spend > 3000:
        label = 'Mid-Tier Customers - moderate spend'
        action = 'Upsell data bundles, targeted promotions'
    else:
        label = 'Low-Engagement Customers - low spend, few transactions'
        action = 'Re-engagement campaigns, cheaper starter plans'
    print(f'Cluster {cluster_id}: avg spend=₦{spend:,.0f}, avg txns={txns:.0f}')
    print(f'  Interpretation: {label}')
    print(f'  Business action: {action}')
    print()

---
## Part 3 - Key Differences: Supervised vs Unsupervised

In [ ]:
# Side-by-side comparison
comparison = {
    'Aspect':       ['Has labels?', 'fit() call',      'Output',             'Can predict new cases?', 'Nigerian use case'],
    'Supervised':   ['YES',         'fit(X, y)',        'Predicted class/value', 'YES - model.predict()',  'Churn prediction'],
    'Unsupervised': ['NO',          'fit(X) - no y!',  'Cluster assignment',    'YES - kmeans.predict()', 'Customer segmentation'],
}
comparison_df = pd.DataFrame(comparison)
comparison_df = comparison_df.set_index('Aspect')
print(comparison_df.to_string())

---
## Summary

**Supervised Learning (churn demo):**
- Had labelled data: `churned = 1 or 0`
- Trained model to predict the label for new customers
- Output: a definite prediction (will churn / will stay)

**Unsupervised Learning (segmentation demo):**
- Had NO labels - only measurements
- Algorithm discovered 3 natural customer groups automatically
- Output: cluster assignments that a human then interprets

**The key rule:**
- Data has a target column → Supervised
- Data has no target column → Unsupervised

**Next - Topic 3:** The ML Workflow. We map out every step from business problem to deployed model.